In [1]:
import pandas as pd
from datasets import Dataset
import os

csv_path = '/kaggle/input/datasets/ananthu017/squad-csv-format/SQuAD_csv.csv'

try:
    print(f"[DEBUG] Membaca file dari: {csv_path}")
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"File tidak ditemukan di {csv_path}")
        
    df = pd.read_csv(csv_path)
    print(f"[DEBUG] Total baris awal: {len(df)}")

    # Bersihkan Data
    df = df.dropna(subset=['context', 'question', 'text', 'answer_start'])
    print(f"[DEBUG] Total baris setelah dropna: {len(df)}")

    # Pastikan Tipe Data
    df['answer_start'] = df['answer_start'].astype(int)
    df['text'] = df['text'].astype(str)

    # Konversi ke Dataset
    dataset = Dataset.from_pandas(df)
    dataset = dataset.remove_columns(["__index_level_0__"]) if "__index_level_0__" in dataset.column_names else dataset
    
    # Split
    dataset = dataset.train_test_split(test_size=0.1)
    print(f"[DEBUG] Pembagian dataset berhasil. Train: {len(dataset['train'])}, Test: {len(dataset['test'])}")
    
except Exception as e:
    print(f"[ERROR] Terjadi kesalahan saat memproses data: {e}")

[DEBUG] Membaca file dari: /kaggle/input/datasets/ananthu017/squad-csv-format/SQuAD_csv.csv
[DEBUG] Total baris awal: 86821
[DEBUG] Total baris setelah dropna: 86818
[DEBUG] Pembagian dataset berhasil. Train: 78136, Test: 8682


In [2]:
from transformers import AutoTokenizer

try:
    print("[DEBUG] Menginisialisasi tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

    def preprocess_function(examples):
        questions = [q.strip() for q in examples["question"]]
        contexts = [c.strip() for c in examples["context"]]
        
        inputs = tokenizer(
            questions,
            contexts,
            max_length=384,
            truncation="only_second",
            return_offsets_mapping=True,
            padding="max_length",
        )

        offset_mapping = inputs.pop("offset_mapping")
        start_positions = []
        end_positions = []

        for i, offset in enumerate(offset_mapping):
            answer = examples["text"][i]
            start_char = examples["answer_start"][i]
            end_char = start_char + len(answer)
            
            sequence_ids = inputs.sequence_ids(i)

            idx = 0
            while sequence_ids[idx] != 1:
                idx += 1
            context_start = idx
            
            idx = len(sequence_ids) - 1
            while sequence_ids[idx] != 1:
                idx -= 1
            context_end = idx

            if offset[context_start][0] > end_char or offset[context_end][1] < start_char:
                start_positions.append(0)
                end_positions.append(0)
            else:
                idx = context_start
                while idx <= context_end and offset[idx][0] <= start_char:
                    idx += 1
                start_positions.append(idx - 1)

                idx = context_end
                while idx >= context_start and offset[idx][1] >= end_char:
                    idx -= 1
                end_positions.append(idx + 1)

        inputs["start_positions"] = start_positions
        inputs["end_positions"] = end_positions
        return inputs

    print("[DEBUG] Memulai proses pemetaan (tokenization) pada dataset...")
    tokenized_datasets = dataset.map(preprocess_function, batched=True)
    print(f"[DEBUG] Kolom yang tersedia setelah tokenisasi: {tokenized_datasets['train'].column_names}")

except Exception as e:
    print(f"[ERROR] Gagal melakukan tokenisasi: {e}")

[DEBUG] Menginisialisasi tokenizer...


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

[DEBUG] Memulai proses pemetaan (tokenization) pada dataset...


Map:   0%|          | 0/78136 [00:00<?, ? examples/s]

Map:   0%|          | 0/8682 [00:00<?, ? examples/s]

[DEBUG] Kolom yang tersedia setelah tokenisasi: ['Unnamed: 0', 'context', 'question', 'id', 'answer_start', 'text', 'input_ids', 'token_type_ids', 'attention_mask', 'start_positions', 'end_positions']


In [3]:
from transformers import AutoModelForQuestionAnswering, TrainingArguments, Trainer
import torch

save_path = "/kaggle/working/sistem-interview-final"

try:
    print("[DEBUG] Memuat model arsitektur Question Answering...")
    model = AutoModelForQuestionAnswering.from_pretrained("distilbert-base-uncased")
    
    # Cek apakah GPU tersedia untuk memastikan training tidak error
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"[DEBUG] Perangkat yang digunakan untuk training: {device.upper()}")
    if device == "cpu":
        print("[WARNING] GPU tidak terdeteksi! Training dengan CPU akan sangat lambat atau gagal.")

    training_args = TrainingArguments(
        output_dir="./qa_interview_model",
        eval_strategy="epoch",           
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=3,
        weight_decay=0.01,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_datasets["train"],
        eval_dataset=tokenized_datasets["test"],
        processing_class=tokenizer,         
    )

    print("[DEBUG] Memulai proses training (Ini mungkin memakan waktu)...")
    trainer.train()
    print("[DEBUG] Training selesai tanpa hambatan!")
    
    # Menyimpan model
    print(f"[DEBUG] Menyimpan model ke {save_path}...")
    trainer.save_model(save_path)
    print("[DEBUG] Model berhasil disimpan!")

except RuntimeError as re:
    print(f"[ERROR CUDA/Runtime] Terjadi kegagalan saat komputasi training: {re}")
except Exception as e:
    print(f"[ERROR Umum] Proses training gagal: {e}")

[DEBUG] Memuat model arsitektur Question Answering...


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForQuestionAnswering LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
qa_outputs.bias         | MISSING    | 
qa_outputs.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[DEBUG] Perangkat yang digunakan untuk training: CUDA
[DEBUG] Memulai proses training (Ini mungkin memakan waktu)...


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


[DEBUG] Training selesai tanpa hambatan!
[DEBUG] Menyimpan model ke /kaggle/working/sistem-interview-final...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[DEBUG] Model berhasil disimpan!


In [4]:
# Simpan model hasil training
trainer.save_model("/kaggle/working/sistem-interview-final")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [5]:
from transformers import pipeline
import os

model_path = "/kaggle/working/sistem-interview-final"

try:
    if not os.path.exists(model_path):
        raise FileNotFoundError(f"[ERROR] Folder model tidak ditemukan di {model_path}. Pastikan proses training berhasil disimpan.")
        
    print(f"[DEBUG] Memuat pipeline dari model lokal: {model_path}...")
    qa_pipeline = pipeline("question-answering", model=model_path, tokenizer=model_path)

    konteks = "Kandidat memiliki pengalaman 3 tahun sebagai Fullstack Developer dengan spesialisasi Backend Engineering. Teknologi yang dikuasai meliputi Go, PHP, dan Java, serta implementasi Clean Architecture."
    pertanyaan = "Apa bahasa pemrograman yang dikuasai kandidat?"

    print("[DEBUG] Menjalankan inferensi jawaban...")
    hasil = qa_pipeline(question=pertanyaan, context=konteks)
    
    print("\n--- HASIL ---")
    print(f"Pertanyaan : {pertanyaan}")
    print(f"Jawaban    : {hasil['answer']}")
    print(f"Skor Yakin : {hasil['score']:.4f}")

except Exception as e:
    print(f"[ERROR] Pipeline inferensi gagal berjalan: {e}")

[DEBUG] Memuat pipeline dari model lokal: /kaggle/working/sistem-interview-final...


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[DEBUG] Menjalankan inferensi jawaban...

--- HASIL ---
Pertanyaan : Apa bahasa pemrograman yang dikuasai kandidat?
Jawaban    : Teknologi
Skor Yakin : 0.0367
